# Figure 12 — your encoding matters more than your acquisition function

One-hot encoding makes every ligand exactly as far from every other. A descriptor encoding puts chemically similar ligands close together, so information transfers between them. Ligand SMILES are verbatim from the EDBO direct-arylation dataset.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

## The ligands

Six of the twelve phosphines in the dataset, with SMILES taken verbatim from `experiment_index.csv`. If the notebook can reach the network it uses all twelve.

> The descriptors below are simple countable features parsed straight from the SMILES string — heavy-atom count, ring-bond count, aromatic fraction and so on. **Shields et al. use DFT-derived steric and electronic descriptors, which are better.** The point of this figure is the contrast with one-hot, and better descriptors only strengthen it.

In [ ]:

import edbo_data as ed

lig = dict(ed.CACHED_LIGANDS)
if ed.available():
    df = ed.load()
    uniq = list(dict.fromkeys(df["Ligand_SMILES"]))
    known = {v: k for k, v in ed.CACHED_LIGANDS.items()}
    lig = {known.get(s, f"ligand {i+1}"): s for i, s in enumerate(uniq)}
    print(f"loaded {len(lig)} ligands from the full dataset")
else:
    print(f"offline — using the {len(lig)} cached ligands")

names, Dmat, keys = ed.descriptor_matrix(lig)
print("descriptors:", keys)
for n_, row in zip(names, Dmat):
    print(f"  {n_:14s}", {k: int(v) for k, v in zip(keys, row)})

In [ ]:

OH = ed.onehot_matrix(names)
d_oh = ed.pairwise(OH)
d_de = ed.pairwise(ed.zscore(Dmat))
scores, var, _ = ed.pca_2d(Dmat)

fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.15),
                         gridspec_kw=dict(wspace=0.42, width_ratios=[1, 1.15]))

im = axes[0].imshow(d_oh, cmap="BuGn", vmin=0, vmax=max(d_oh.max(), d_de.max()))
axes[0].set_xticks(range(len(names))); axes[0].set_yticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=90, fontsize=8.5)
axes[0].set_yticklabels(names, fontsize=8.5)
axes[0].set_title("one-hot: every pair\nexactly as far apart", loc="left",
                  fontsize=10.5, color=style.RED)
for sp in axes[0].spines.values():
    sp.set_visible(False)
off = d_oh[~np.eye(len(names), dtype=bool)]

ax = axes[1]
ax.scatter(scores[:, 0], scores[:, 1], s=85, c=style.TEAL, edgecolor="white",
           linewidth=1.1, zorder=5)
for (x, y), n_ in zip(scores, names):
    ax.annotate(n_, (x, y), textcoords="offset points", xytext=(8, 5),
                fontsize=9.0, color=style.INK)
ax.set_xlabel(f"PC1 ({100*var[0]:.0f}% of variance)")
ax.set_ylabel(f"PC2 ({100*var[1]:.0f}%)")
ax.set_title("descriptors: chemical\nfamilies separate", loc="left",
             fontsize=10.5, color=style.TEAL)
ax.margins(0.30)
style.save(fig, "fig_12_categorical_encoding", OUT)
print("one-hot: every off-diagonal distance =", round(off.mean(), 4),
      "  sd =", round(off.std(), 6), " (identical by construction)")
print("descriptor distances range", round(d_de[d_de > 0].min(), 2), "-",
      round(d_de.max(), 2))